In [1]:
grade_url = {
    10: "https://loigiaihay.com/lop-10.html",
    11: "https://loigiaihay.com/lop-11.html",
    12: "https://loigiaihay.com/lop-12.html",
}

In [ ]:
# Cell 2: imports + config + directory layout
import os
import re
import json
import time
import shutil
import unicodedata
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
from urllib.parse import urljoin, urlparse, parse_qs

import requests
from bs4 import BeautifulSoup

# Optional: PDF crop support (header/footer removal)
try:
    from pypdf import PdfReader, PdfWriter
    HAS_PYPDF = True
except Exception:
    HAS_PYPDF = False

BASE_URL = "https://loigiaihay.com"
PROJECT_ROOT = Path.cwd().parent.parent  # notebooks/exploration/ -> project root
DATA_DIR = PROJECT_ROOT / "data"
META_DIR = PROJECT_ROOT / "notebooks" / "data"
META_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
}

# Bắt các loại tài liệu cần lấy
DOC_KEYWORDS = {
    "sgk": ["sgk", "sach-giao-khoa", "sách giáo khoa"],
    "sbt": ["sbt", "sach-bai-tap", "sách bài tập", "bai-tap"],
    "de_thi": ["de-thi", "đề thi", "de kiem tra", "đề kiểm tra"],
    "chuyen_de": ["chuyen-de", "chuyên đề"],
    "trac_nghiem": ["trac-nghiem", "trắc nghiệm"],
}

SUBJECT_MAP = {
    "toan": ["toan", "đại số", "hinh-hoc"],
    "ngu-van": ["van", "ngu-van", "ngữ văn"],
    "vat-li": ["vat-li", "vật lí", "vật lý", "ly"],
    "hoa-hoc": ["hoa", "hoa-hoc", "hóa"],
    "sinh-hoc": ["sinh", "sinh-hoc"],
    "lich-su": ["lich-su", "lịch sử", "su"],
    "dia-li": ["dia-li", "địa lí", "địa lý"],
    "tieng-anh": ["tieng-anh", "tiếng anh", "english"],
    "tin-hoc": ["tin-hoc", "tin học"],
    "cong-nghe": ["cong-nghe", "công nghệ"],
    "gdcd": ["gdcd", "kinh-te", "phap-luat", "giáo dục kinh tế"],
    "gdqp-an": ["quoc-phong", "an-ninh", "gdqp"],
    "hoat-dong-trai-nghiem": ["trai-nghiem", "huong-nghiep"],
}


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return re.sub(r"-+", "-", text).strip("-")


def ensure_layout(base_dir: Path) -> None:
    """Create data/lop-10|11|12/{sgk,sbt} structure if missing."""
    for grade in (10, 11, 12):
        for bucket in ("sgk", "sbt"):
            (base_dir / f"lop-{grade}" / bucket).mkdir(parents=True, exist_ok=True)


ensure_layout(DATA_DIR)
print(f"Data layout ready under: {DATA_DIR}")
print("Expected buckets: sgk, sbt")

Data layout ready under: /home/sakana/Code/ManifoldsTeam/AIServices/data
Expected buckets: sgk, sbt


In [3]:
# Cell 3: move existing PDFs into new sgk/sbt buckets (migration step)

def guess_bucket_from_name(name: str) -> str:
    t = normalize_text(name)
    if any(k in t for k in ["sbt", "de-thi", "de-kiem-tra", "chuyen-de", "trac-nghiem", "bai-tap"]):
        return "sbt"
    return "sgk"


def migrate_existing_pdfs(base_dir: Path) -> dict:
    moved = 0
    skipped = 0
    per_grade = {}

    for grade in (10, 11, 12):
        grade_dir = base_dir / f"lop-{grade}"
        if not grade_dir.exists():
            continue

        per_grade[f"lop-{grade}"] = {"moved": 0, "skipped": 0}

        for subject_dir in grade_dir.iterdir():
            if not subject_dir.is_dir() or subject_dir.name in {"sgk", "sbt"}:
                continue

            for pdf in subject_dir.glob("*.pdf"):
                bucket = guess_bucket_from_name(pdf.name)
                target_dir = grade_dir / bucket / subject_dir.name
                target_dir.mkdir(parents=True, exist_ok=True)
                target = target_dir / pdf.name

                if target.exists() and target.stat().st_size > 0:
                    skipped += 1
                    per_grade[f"lop-{grade}"]["skipped"] += 1
                    continue

                shutil.move(str(pdf), str(target))
                moved += 1
                per_grade[f"lop-{grade}"]["moved"] += 1

    return {"moved": moved, "skipped": skipped, "detail": per_grade}


migration_stats = migrate_existing_pdfs(DATA_DIR)
print(json.dumps(migration_stats, ensure_ascii=False, indent=2))

{
  "moved": 242,
  "skipped": 0,
  "detail": {
    "lop-10": {
      "moved": 66,
      "skipped": 0
    },
    "lop-11": {
      "moved": 101,
      "skipped": 0
    },
    "lop-12": {
      "moved": 75,
      "skipped": 0
    }
  }
}


In [4]:
# Cell 4: crawler + metadata extraction + google drive download URL parsing

@dataclass
class DocItem:
    grade: int
    subject: str
    bucket: str           # sgk|sbt (requirement)
    doc_type: str         # sgk|sbt|de_thi|chuyen_de|trac_nghiem
    title: str
    page_url: str
    download_url: str
    file_id: Optional[str]


def detect_grade(text: str) -> Optional[int]:
    for g in (10, 11, 12):
        if re.search(rf"(^|[^0-9]){g}([^0-9]|$)", text):
            return g
    return None


def detect_doc_type(text: str) -> str:
    t = text.lower()
    for dt, kws in DOC_KEYWORDS.items():
        if any(k in t for k in kws):
            return dt
    return "sgk"


def detect_subject(text: str) -> str:
    t = text.lower()
    for subject, kws in SUBJECT_MAP.items():
        if any(k in t for k in kws):
            return subject
    return "khac"


def bucket_from_doc_type(doc_type: str) -> str:
    # Theo yêu cầu chỉ có 2 bucket: sgk, sbt
    if doc_type == "sgk":
        return "sgk"
    return "sbt"


def to_abs(url: str) -> str:
    return url if url.startswith("http") else urljoin(BASE_URL, url)


def extract_drive_file_id(url: str) -> Optional[str]:
    # Pattern 1: /file/d/<id>/view
    m = re.search(r"/file/d/([a-zA-Z0-9_-]+)", url)
    if m:
        return m.group(1)

    # Pattern 2: open?id=<id>
    query = parse_qs(urlparse(url).query)
    if "id" in query and query["id"]:
        return query["id"][0]
    return None


def normalize_drive_download(url: str) -> tuple[str, Optional[str]]:
    fid = extract_drive_file_id(url)
    if fid:
        return f"https://drive.google.com/uc?export=download&id={fid}", fid
    return url, None


def is_candidate_link(href: str, text: str) -> bool:
    hay = f"{href} {text}".lower()
    return any(k in hay for k in [
        "pdf", "sbt", "sach-bai-tap", "de-thi", "chuyen-de", "trac-nghiem", "bai-tap", "sach-giao-khoa", "sgk"
    ])


def collect_document_pages(max_pages_per_grade: int = 100) -> list[str]:
    session = requests.Session()
    session.headers.update(HEADERS)

    pages = set()
    for grade, root in grade_url.items():
        print(f"Collecting seed links from lớp {grade} ...")
        resp = session.get(root, timeout=25)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "lxml")

        for a in soup.find_all("a", href=True):
            href = to_abs(a["href"])
            text = a.get_text(" ", strip=True)
            if "loigiaihay.com" not in href:
                continue
            if not is_candidate_link(href, text):
                continue

            if re.search(r"-c\d+\.html$", href) or re.search(r"-a\d+\.html$", href):
                pages.add(href)

        # Crawl one level deeper for category pages
        cat_pages = [p for p in pages if re.search(r"-c\d+\.html$", p) and detect_grade(p) in (None, grade)]
        for p in cat_pages[:max_pages_per_grade]:
            try:
                r = session.get(p, timeout=25)
                s = BeautifulSoup(r.text, "lxml")
                for a in s.find_all("a", href=True):
                    href = to_abs(a["href"])
                    text = a.get_text(" ", strip=True)
                    if "loigiaihay.com" not in href:
                        continue
                    if re.search(r"-a\d+\.html$", href) and is_candidate_link(href, text):
                        pages.add(href)
            except Exception:
                continue

    return sorted(pages)


def parse_document_page(url: str, default_grade: Optional[int] = None) -> list[DocItem]:
    session = requests.Session()
    session.headers.update(HEADERS)

    r = session.get(url, timeout=25)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    title = (soup.find("h1").get_text(" ", strip=True) if soup.find("h1") else "")
    title_norm = normalize_text(title)
    source_text = f"{url} {title}".lower()

    grade = detect_grade(source_text) or default_grade
    if grade not in (10, 11, 12):
        return []

    doc_type = detect_doc_type(source_text)
    subject = detect_subject(source_text)
    bucket = bucket_from_doc_type(doc_type)

    out: list[DocItem] = []
    seen_dl = set()

    for a in soup.find_all("a", href=True):
        href = to_abs(a["href"])
        text = a.get_text(" ", strip=True)
        h_low = href.lower()
        t_low = text.lower()

        is_dl = (
            "drive.google.com" in h_low
            or h_low.endswith(".pdf")
            or "download file" in t_low
            or t_low == "tải về"
            or " pdf" in t_low
        )
        if not is_dl:
            continue

        dl, fid = normalize_drive_download(href)
        if dl in seen_dl:
            continue
        seen_dl.add(dl)

        out.append(
            DocItem(
                grade=grade,
                subject=subject,
                bucket=bucket,
                doc_type=doc_type,
                title=title,
                page_url=url,
                download_url=dl,
                file_id=fid,
            )
        )

    # fallback: raw source contains direct pdf url
    if not out:
        pdf_urls = re.findall(r"https?://[^\s\"'<>]+\.pdf", r.text)
        for pu in pdf_urls:
            if pu in seen_dl:
                continue
            seen_dl.add(pu)
            out.append(
                DocItem(
                    grade=grade,
                    subject=subject,
                    bucket=bucket,
                    doc_type=doc_type,
                    title=title,
                    page_url=url,
                    download_url=pu,
                    file_id=None,
                )
            )

    return out


print("Crawler helpers ready")

Crawler helpers ready


In [5]:
# Cell 5: collect all document download entries and save manifest

all_pages = collect_document_pages(max_pages_per_grade=120)
print(f"Candidate pages: {len(all_pages)}")

items: list[DocItem] = []
for idx, page in enumerate(all_pages, 1):
    try:
        parsed = parse_document_page(page)
        if parsed:
            items.extend(parsed)
        if idx % 40 == 0:
            print(f"  parsed {idx}/{len(all_pages)} pages -> {len(items)} download items")
        time.sleep(0.15)
    except Exception:
        continue

# de-duplicate by download_url
uniq = {}
for it in items:
    uniq[it.download_url] = it
items = list(uniq.values())

manifest_path = META_DIR / "loigiaihay_download_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump([asdict(x) for x in items], f, ensure_ascii=False, indent=2)

print(f"Unique download items: {len(items)}")
print(f"Manifest saved: {manifest_path}")
print("Sample:")
for s in items[:5]:
    print(f"- lop-{s.grade}/{s.bucket}/{s.subject}: {s.title[:80]}")

Candidate pages: 5392
  parsed 40/5392 pages -> 0 download items
  parsed 80/5392 pages -> 0 download items
  parsed 120/5392 pages -> 0 download items
  parsed 160/5392 pages -> 0 download items
  parsed 200/5392 pages -> 0 download items
  parsed 240/5392 pages -> 0 download items
  parsed 280/5392 pages -> 0 download items
  parsed 320/5392 pages -> 0 download items
  parsed 360/5392 pages -> 0 download items
  parsed 400/5392 pages -> 0 download items
  parsed 440/5392 pages -> 0 download items
  parsed 480/5392 pages -> 0 download items
  parsed 520/5392 pages -> 0 download items
  parsed 560/5392 pages -> 0 download items
  parsed 600/5392 pages -> 0 download items
  parsed 640/5392 pages -> 0 download items
  parsed 680/5392 pages -> 0 download items
  parsed 720/5392 pages -> 0 download items
  parsed 760/5392 pages -> 0 download items
  parsed 800/5392 pages -> 0 download items
  parsed 840/5392 pages -> 0 download items
  parsed 880/5392 pages -> 1 download items
  parsed 920

KeyboardInterrupt: 

In [ ]:
# Cell 6: file naming + download + optional crop header/footer

def infer_filename(item: DocItem) -> str:
    title_slug = normalize_text(item.title)[:120]
    base = f"lop-{item.grade}_{item.doc_type}_{item.subject}_{title_slug}"
    if item.file_id:
        base += f"_{item.file_id[:8]}"
    return base + ".pdf"


def crop_pdf_header_footer(src_path: Path, dst_path: Path, top_ratio: float = 0.14, bottom_ratio: float = 0.06) -> bool:
    """Crop top/bottom region to remove site banner/footer in scanned PDFs."""
    if not HAS_PYPDF:
        return False

    reader = PdfReader(str(src_path))
    writer = PdfWriter()

    for page in reader.pages:
        box = page.mediabox
        x0, y0, x1, y1 = float(box.left), float(box.bottom), float(box.right), float(box.top)
        h = y1 - y0
        new_bottom = y0 + (h * bottom_ratio)
        new_top = y1 - (h * top_ratio)

        page.cropbox.lower_left = (x0, new_bottom)
        page.cropbox.upper_right = (x1, new_top)
        page.mediabox.lower_left = (x0, new_bottom)
        page.mediabox.upper_right = (x1, new_top)
        writer.add_page(page)

    with open(dst_path, "wb") as f:
        writer.write(f)
    return True


def download_file(url: str, out_path: Path, timeout: int = 120) -> str:
    if out_path.exists() and out_path.stat().st_size > 2000:
        return "skipped"

    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with requests.get(url, headers=HEADERS, timeout=timeout, stream=True, allow_redirects=True) as r:
            if r.status_code != 200:
                return f"http_{r.status_code}"

            ctype = (r.headers.get("content-type") or "").lower()
            if "html" in ctype and "pdf" not in ctype:
                return "not_pdf"

            with open(out_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        if out_path.stat().st_size < 2000:
            out_path.unlink(missing_ok=True)
            return "too_small"

        return "ok"
    except Exception:
        return "failed"


def process_downloads(manifest_file: Path, crop_enabled: bool = True, top_ratio: float = 0.14, bottom_ratio: float = 0.06, limit: Optional[int] = None):
    with open(manifest_file, "r", encoding="utf-8") as f:
        rows = json.load(f)

    stats = {"ok": 0, "skipped": 0, "failed": 0, "cropped": 0, "not_pdf": 0, "too_small": 0}
    failures = []

    to_process = rows if limit is None else rows[:limit]

    for i, row in enumerate(to_process, 1):
        item = DocItem(**row)
        file_name = infer_filename(item)

        target_dir = DATA_DIR / f"lop-{item.grade}" / item.bucket / item.subject
        raw_path = target_dir / file_name

        status = download_file(item.download_url, raw_path)
        if status in stats:
            stats[status] += 1
        elif status.startswith("http_"):
            stats["failed"] += 1
        else:
            stats["failed"] += 1

        if status == "ok" and crop_enabled and HAS_PYPDF:
            cleaned = raw_path.with_name(raw_path.stem + "_clean.pdf")
            try:
                if crop_pdf_header_footer(raw_path, cleaned, top_ratio=top_ratio, bottom_ratio=bottom_ratio):
                    raw_path.unlink(missing_ok=True)
                    cleaned.rename(raw_path)
                    stats["cropped"] += 1
            except Exception:
                pass

        if status not in ("ok", "skipped"):
            failures.append({
                "idx": i,
                "status": status,
                "url": item.download_url,
                "page": item.page_url,
                "title": item.title,
            })

        if i % 25 == 0:
            print(f"Processed {i}/{len(to_process)} | ok={stats['ok']} skipped={stats['skipped']} failed={stats['failed']}")

        time.sleep(0.15)

    fail_path = META_DIR / "loigiaihay_download_failures.json"
    with open(fail_path, "w", encoding="utf-8") as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)

    return stats, failures


print("Download pipeline ready. Run Cell 7 to execute.")

In [ ]:
# Cell 7: execute download (dry run first with limit), then full run
manifest_path = META_DIR / "loigiaihay_download_manifest.json"

if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}. Run Cell 5 first.")

# Gợi ý: chạy thử với limit=30 trước, sau đó để None để tải toàn bộ.
stats, failures = process_downloads(
    manifest_file=manifest_path,
    crop_enabled=True,
    top_ratio=0.14,
    bottom_ratio=0.06,
    limit=30,
)

print("\n=== DOWNLOAD SUMMARY ===")
print(json.dumps(stats, ensure_ascii=False, indent=2))
print(f"Failures: {len(failures)}")
if failures:
    print("First 5 failures:")
    for f in failures[:5]:
        print(f"- [{f['status']}] {f['title'][:80]}")

In [ ]:
# Cell 8: verify folder structure and sample outputs

def summarize_tree(base_dir: Path):
    report = {}
    for grade in (10, 11, 12):
        gdir = base_dir / f"lop-{grade}"
        if not gdir.exists():
            continue
        report[f"lop-{grade}"] = {}
        for bucket in ("sgk", "sbt"):
            bdir = gdir / bucket
            if not bdir.exists():
                report[f"lop-{grade}"][bucket] = 0
                continue
            count = len(list(bdir.rglob("*.pdf")))
            report[f"lop-{grade}"][bucket] = count
    return report

summary = summarize_tree(DATA_DIR)
print(json.dumps(summary, ensure_ascii=False, indent=2))

print("\nSample files:")
for g in (10, 11, 12):
    for b in ("sgk", "sbt"):
        folder = DATA_DIR / f"lop-{g}" / b
        files = list(folder.rglob("*.pdf"))[:3]
        for f in files:
            print(f"- {f.relative_to(DATA_DIR)}")